<a href="https://colab.research.google.com/github/jyothsnareddym/BIS_LAB/blob/main/1BF24CS180_w2_PSO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np


class DroneRoutePSO:

    def __init__(
        self, start, dest, obstacles, num_waypoints=5, swarm_size=40, max_iter=100
    ):
        self.start = np.array(start)
        self.dest = np.array(dest)
        self.obstacles = obstacles  # List of [[x, y, radius], ...]
        self.num_waypoints = num_waypoints
        self.swarm_size = swarm_size
        self.max_iter = max_iter

        # Airspace Map Limits
        self.min_bound, self.max_bound = 0, 100

        self.w = 0.7
        self.c1 = 1.5
        self.c2 = 1.5

        self.X = np.zeros((swarm_size, num_waypoints, 2))
        for i in range(swarm_size):
            for w_idx in range(num_waypoints):
                fraction = (w_idx + 1) / (num_waypoints + 1)
                base = self.start + fraction * (self.dest - self.start)
                self.X[i, w_idx] = base + np.random.uniform(-10, 10, size=2)

        # Initialize adjustment speeds (velocities) randomly
        self.V = np.random.uniform(-2, 2, size=(swarm_size, num_waypoints, 2))

        # Initialize Personal Bests (pbest) to their starting layouts
        self.pbest = np.copy(self.X)
        self.pbest_fit = np.array(
            [self.calculate_fitness(path) for path in self.pbest]
        )

        # Initialize Global Best (gbest) to the single best initial path layout
        gbest_idx = np.argmin(self.pbest_fit)
        self.gbest = np.copy(self.pbest[gbest_idx])
        self.gbest_fit = self.pbest_fit[gbest_idx]

    def check_collision(self, p1, p2, obstacle):
        """Helper to check if a single path segment clips a circular obstacle."""
        cx, cy, r = obstacle
        p1, p2 = np.array(p1), np.array(p2)
        line_vec = p2 - p1
        circle_vec = np.array([cx, cy]) - p1
        line_len = np.sum(line_vec**2)

        if line_len == 0:
            return np.sum(circle_vec**2) <= r**2

        t = np.clip(np.dot(circle_vec, line_vec) / line_len, 0.0, 1.0)
        closest_point = p1 + t * line_vec
        distance = np.sqrt(np.sum((closest_point - [cx, cy]) ** 2))
        return distance <= r

    def calculate_fitness(self, waypoints):
        """Fitness Function: Computes (Total Length + Crash Penalties)"""
        full_path = np.vstack([self.start, waypoints, self.dest])

        # 1. Total Distance Check
        distance = 0.0
        for i in range(len(full_path) - 1):
            distance += np.linalg.norm(full_path[i + 1] - full_path[i])

        # 2. Obstacle Penalty Check
        penalty = 0
        for i in range(len(full_path) - 1):
            for obs in self.obstacles:
                if self.check_collision(full_path[i], full_path[i + 1], obs):
                    penalty += 100000  # Massive penalty if it hits

        return distance + penalty

    def run_cycles(self):
        """STEP 2: THE CONTINUOUS OPTIMIZATION LOOP (The Cycles)"""
        for iteration in range(self.max_iter):
            for i in range(self.swarm_size):

                # --- 1. REARRANGE PATH ---
                r1 = np.random.rand(self.num_waypoints, 2)
                r2 = np.random.rand(self.num_waypoints, 2)

                # Update Velocity vector
                self.V[i] = (
                    (self.w * self.V[i])
                    + (self.c1 * r1 * (self.pbest[i] - self.X[i]))
                    + (self.c2 * r2 * (self.gbest - self.X[i]))
                )

                # Keep rearrangement speeds controlled
                self.V[i] = np.clip(self.V[i], -4, 4)

                # Update Position (Physically shifts the line coordinates)
                self.X[i] += self.V[i]
                self.X[i] = np.clip(self.X[i], self.min_bound, self.max_bound)

                # --- 2. CHECK FITNESS & UPDATE MEMORIES ---
                current_fit = self.calculate_fitness(self.X[i])

                # Update individual path memory (pbest)
                if current_fit < self.pbest_fit[i]:
                    self.pbest[i] = np.copy(self.X[i])
                    self.pbest_fit[i] = current_fit

                    # Update shared swarm memory (gbest)
                    if current_fit < self.gbest_fit:
                        self.gbest = np.copy(self.X[i])
                        self.gbest_fit = current_fit

        # STEP 3: Return final optimized non-deviating corridor coordinates
        final_route = np.vstack([self.start, self.gbest, self.dest])
        return final_route, self.gbest_fit


if __name__ == "__main__":
    # Filled coordinates [X, Y] for takeoff and landing
    launch_pad = [5, 5]
    landing_pad = [95, 95]

    # Concrete obstacle layout metrics: [[Center-X, Center-Y, Radius]]
    city_obstacles = [
        [30, 40, 12],  # Building 1
        [50, 50, 15],  # Building 2
        [70, 60, 10],  # Building 3
    ]

    # Initialize router with a swarm of 40 alternate routes
    router = DroneRoutePSO(
        start=launch_pad,
        dest=landing_pad,
        obstacles=city_obstacles,
        num_waypoints=5,
        swarm_size=40,
        max_iter=120,
    )

    # Run the cycle loop
    best_path, final_score = router.run_cycles()

    print("--- OPTIMIZATION COMPLETE ---")
    print(f"Final Path Score: {final_score:.2f}")
    print("Coordinates Matrix along the path:\n", best_path)


--- OPTIMIZATION COMPLETE ---
Final Path Score: 200128.61
Coordinates Matrix along the path:
 [[ 5.          5.        ]
 [26.7256862  21.88494642]
 [38.89851698 31.74630763]
 [48.27957506 45.48879704]
 [64.58328415 69.89670988]
 [78.58595244 81.74220732]
 [95.         95.        ]]
